In [1]:
import torch
from datasets import Audio
import librosa

from transformers import WhisperProcessor, WhisperForConditionalGeneration


/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

In [3]:
def load_audio_16k(path: str):
    audio, sr = librosa.load(path, sr=16000, mono=True)
    return audio, sr

In [4]:
device = get_device()
print(f"[info] device = {device}")

[info] device = mps


# Basic Setup

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", task="transcribe", language=None)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny", task="transcribe", language=None).to(device)
model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 384)
      (layers): ModuleList(
        (0-3): 4 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=True)
            (q_proj): Linear(in_features=384, out_features=384, bias=True)
            (out_proj): Linear(in_features=384, out_features=384, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          

In [11]:
audio_path = "../../data/1_02_2026_tcc_cv/clips/Daily schedule_tcc_dato1239_tsim1256_IGS0229_2015-12-8_MM_3_0038990_0042314.mp3" 
audio, sr = librosa.load(audio_path, sr=16000, mono=True)

inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
input_features = inputs.input_features.to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)

text = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(text)

 Cassisa, I'm going to call you in the right.


# Evaluation pipeline

## Load dataset

In [45]:
from datasets import load_dataset, Features, Value, Audio

features = Features({
    "path": Value("string"),
    "sentence": Value("string"),
    "eng": Value("string"),
    "sw": Value("string"),
})

ds = load_dataset(
    "csv",
    data_files={
        "validation": "../../data/1_02_2026_tcc_cv/valid_dataset.tsv",
    },
    delimiter="\t",
    features=features,
)

print('VALIDATION', ds["validation"].column_names)

def fix_path(batch):
    batch["path"] = "/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/data/1_02_2026_tcc_cv/" + batch["path"]
    return batch

ds = ds.map(fix_path)

from datasets import Audio
ds = ds.cast_column("path", Audio(sampling_rate=16000))
ds = ds.with_format("numpy")
ds = ds.rename_column("path", "audio")

Generating validation split: 3202 examples [00:00, 203845.57 examples/s]


VALIDATION ['path', 'sentence', 'eng', 'sw']


Map: 100%|██████████| 3202/3202 [00:00<00:00, 44262.03 examples/s]


In [46]:
from IPython.display import Audio

sample = ds["validation"][0]
Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])

## Features extraction

In [47]:
from transformers import WhisperProcessor, WhisperTokenizer, WhisperFeatureExtractor

processor = WhisperProcessor.from_pretrained('openai/whisper-tiny', language="Swahili", task='transcribe')
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="Swahili", task="transcribe", padding='longest')
feature_extractor = WhisperFeatureExtractor.from_pretrained('openai/whisper-tiny')

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate = audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

preprocessed_ds = ds.map(prepare_dataset, num_proc=4)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map (num_proc=4): 100%|██████████| 3202/3202 [00:11<00:00, 282.73 examples/s]


In [48]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [49]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [50]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
        "combined": 0.5 * wer + 0.5 * cer,
    }

In [66]:
import os
import pandas as pd
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

def eval_model(model_path: str, out_dir: str, dataset, split: str = "test", bs: int = 8):
    os.makedirs(out_dir, exist_ok=True)
    device = get_device()

    model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()

    model.generation_config.task = "transcribe"
    model.generation_config.language = "Swahili" 

    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        per_device_eval_batch_size=bs,
        predict_with_generate=True,
        report_to=[],
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=feature_extractor, 
    )

    pred = trainer.predict(dataset)
    print("METRICS:", pred.metrics)

    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    ref_str  = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    df = pd.DataFrame({"prediction": pred_str, "reference": ref_str})
    df.to_csv(os.path.join(out_dir, f"predictions_{split}.csv"), index=False)

    with open(os.path.join(out_dir, f"metrics_{split}.json"), "w", encoding="utf-8") as f:
        import json
        json.dump(pred.metrics, f, ensure_ascii=False, indent=2)

    return pred.metrics

In [67]:

small_test = preprocessed_ds["validation"].select(range(10))

ASMJ_MODEL = "../../models/whisper-tiny-asmjeeg-2026-2000/checkpoint-2000"
SWAH_MODEL = "../../models/whisper-tiny-sw-2026/checkpoint-2000"

m1 = eval_model(ASMJ_MODEL, out_dir="./eval_out/asmjeeg_on_sw_test", dataset=small_test, split="validation", bs=8)
m2 = eval_model(SWAH_MODEL, out_dir="./eval_out/swahili_on_sw_test", dataset=small_test, split="validation", bs=8)

print("ASMJ:", m1)
print("SWAH:", m2)

/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 2/2 [00:03<00:00,  1.51s/it]


METRICS: {'test_loss': 3.965330123901367, 'test_wer': 1101.1904761904764, 'test_cer': 570.1030927835052, 'test_combined': 835.6467844869908, 'test_runtime': 27.9763, 'test_samples_per_second': 0.357, 'test_steps_per_second': 0.071}


100%|██████████| 2/2 [00:00<00:00,  2.45it/s]


METRICS: {'test_loss': 5.649153232574463, 'test_wer': 109.52380952380953, 'test_cer': 67.83505154639175, 'test_combined': 88.67943053510064, 'test_runtime': 2.1802, 'test_samples_per_second': 4.587, 'test_steps_per_second': 0.917}
ASMJ: {'test_loss': 3.965330123901367, 'test_wer': 1101.1904761904764, 'test_cer': 570.1030927835052, 'test_combined': 835.6467844869908, 'test_runtime': 27.9763, 'test_samples_per_second': 0.357, 'test_steps_per_second': 0.071}
SWAH: {'test_loss': 5.649153232574463, 'test_wer': 109.52380952380953, 'test_cer': 67.83505154639175, 'test_combined': 88.67943053510064, 'test_runtime': 2.1802, 'test_samples_per_second': 4.587, 'test_steps_per_second': 0.917}


In [ ]:
ASMJ_MODEL = "../../models/whisper-tiny-asmjeeg-2026-2000/checkpoint-2000"
SWAH_MODEL = "../../models/whisper-tiny-sw-2026/checkpoint-2000"

m1 = eval_model(ASMJ_MODEL, out_dir="./eval_out/asmjeeg_on_sw_test", dataset=preprocessed_ds['validation'], bs=8)
m2 = eval_model(SWAH_MODEL, out_dir="./eval_out/swahili_on_sw_test", dataset=preprocessed_ds['validation'], bs=8)

print("ASMJ:", m1)
print("SWAH:", m2)

/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
  1%|          | 3/401 [00:37<1:13:12, 11.04s/it]

KeyboardInterrupt: 